# Python Data Pipeline Engineering Lab
## Omnichannel Retail Data Warehouse (Incremental & Idempotent ETL Pipeline)
---
### สารบัญขั้นตอนการทำงาน
1. **Task 1:** Pipeline Configuration & Safe Extraction
2. **Task 2:** Transformation, Data Quality (DQ) & Quarantine Engine
3. **Task 3:** Star Schema Data Warehouse Modeling & Loading
4. **Task 4:** Idempotency & Incremental Loading Demonstration (4 Runs)
5. **Task 5:** Orchestration, KPI Summary & Analytical Queries

In [ ]:
import os
import sys
import logging
import sqlite3
from dataclasses import dataclass, field
from datetime import datetime
from typing import List, Dict, Tuple, Optional
import pandas as pd
import numpy as np

from pipeline import PipelineConfig, run_pipeline, init_database, load_dimensions
print('Environment and Pipeline modules successfully imported!')

## 1. รัน Data Pipeline ครบทุกขั้นตอน (Task 1 - Task 5)
ฟังก์ชัน `run_pipeline` จะทำการ:
- สร้าง SQLite Star Schema (`retail_dw.db`)
- โหลด Dimensions (`dim_customer`, `dim_product`, `dim_date`)
- ประมวลผล 4 รอบ: `batch_1` -> `batch_1 (rerun)` -> `batch_2` -> `batch_3`
- แยกข้อมูล Quarantine ลง `quarantine.csv` และบันทึก Log ลง `pipeline_run_log.csv`

In [ ]:
config = PipelineConfig()
results = run_pipeline(config)

## 2. ตรวจสอบประวัติการรัน (Pipeline Run Log)
ตรวจสอบว่า `rows_read = rows_valid + rows_rejected` และการรันซ้ำในรอบที่ 2 เพิ่ม 0 แถว

In [ ]:
conn = sqlite3.connect('retail_dw.db')
df_log = pd.read_sql('SELECT * FROM pipeline_run_log;', conn)
df_log

## 3. ตรวจสอบข้อมูล Quarantine และเหตุผลความผิดปกติ (Reason Codes)

In [ ]:
df_quarantine = pd.read_sql('''
    SELECT reason_code, COUNT(*) AS count
    FROM quarantine_records
    GROUP BY reason_code
    ORDER BY count DESC;
''', conn)
df_quarantine

## 4. Star Schema Validation & Analytics Queries
ทดสอบ JOIN Star Schema: Fact Table กับ 3 Dimension Tables

In [ ]:
df_analytics = pd.read_sql('''
    SELECT 
        dp.category,
        fs.sales_channel,
        COUNT(fs.order_id) AS total_orders,
        SUM(fs.quantity) AS total_quantity,
        ROUND(SUM(fs.gross_amount), 2) AS total_gross_sales,
        ROUND(SUM(fs.net_amount), 2) AS total_net_sales
    FROM fact_sales fs
    JOIN dim_product dp ON fs.product_key = dp.product_key
    GROUP BY dp.category, fs.sales_channel
    ORDER BY total_net_sales DESC;
''', conn)
df_analytics

## 5. ตรวจสอบ Acceptance Tests ทั้งหมด

In [ ]:
cursor = conn.cursor()
# 1. Check duplicate order_id
cursor.execute('SELECT COUNT(*), COUNT(DISTINCT order_id) FROM fact_sales;')
tot, unq = cursor.fetchone()
print(f'[PASS] Uniqueness: {tot} rows, {unq} unique order_id')

# 2. Check foreign keys
cursor.execute('SELECT COUNT(*) FROM fact_sales fs LEFT JOIN dim_customer dc ON fs.customer_key = dc.customer_key WHERE dc.customer_key IS NULL;')
print(f'[PASS] Orphan customer foreign keys: {cursor.fetchone()[0]}')

# 3. Check non-negative constraints
cursor.execute('SELECT COUNT(*) FROM fact_sales WHERE quantity <= 0 OR unit_price <= 0 OR net_amount < 0;')
print(f'[PASS] Constraint violations: {cursor.fetchone()[0]}')

conn.close()